# Wake Word "Letícia" para Home Assistant (v20 — Fix Sample Rate & TF Crash)

## Correções integradas:
| # | Correção | Impacto |
|---|----------|---------|
| 1 | PyTorch instalado **sem torchvision** | Evita `BeartypeDecorHintForwardRefException` via `onnxscript.values.ParamSchema` |
| 2 | `pronouncing` adicionado às deps | Fix `ModuleNotFoundError: No module named 'pronouncing'` em `data.py:27` |
| 3 | Todas as deps explícitas (não rely em `--no-deps` cascade) | Fix `ModuleNotFoundError: torchmetrics`, speechbrain, etc. |
| 4 | Verificação e correção de sample rate do FMA (→16kHz) | Fix `ValueError: Clip does not have the correct sample rate` |
| 5 | Smart cleanup de features: remove apenas `.npy`, preserva clips | Evita `ValueError: high <= 0` ao reprocessar após crash parcial |
| 6 | Pre-flight check antes do Etapa 5 | Falha rápida com mensagem clara antes de rodar `train.py` |
| 7 | Cada etapa gera `output/etapa-XX/tree.txt` | Rastreabilidade completa do estado do filesystem |
| 8 | Diagnóstico aprimorado com verificação de todas as versões | Detecta problemas antes de cada fase |
| 9 | `pip_instalar()` helper — warnings ≠ erros | `⚠️ aviso` para conflitos Colab, `❌ erro` só para falhas reais |
| 10 | `onnxruntime>=1.17.0` e `tensorflow-cpu>=2.16.0` | Versões pinadas removidas do PyPI para Python 3.12 |
| 11 | `speexdsp-ns` e `huggingface-hub>=1.3.0` adicionados | Deps faltantes do openwakeword 0.6.0 |
| 12 | FMA com `trust_remote_code=True` + try/except por clip | Fix `ValueError: Cannot seek streaming HTTP file` — FMA é opcional |
| 13 | Etapa 5 pre-flight usa `find_spec` em vez de `__import__` | Fix `BeartypeDecorHintForwardRefException` no pre-flight |
| 14 | `transformers<4.46.0` (era `<5.0.0`) | `flex_attention.py` introduzido no 4.46 — pin evita o import chain |
| 15 | Patch `onnxscript/values.py` em disco | Fix subprocess: injeta `class ParamSchema` antes de rodar `train.py` |
| 16 | **`protobuf==4.25.5`** no Etapa 1a | `transformers.image_transforms` importa TF que precisa de `google.protobuf.runtime_version` |
| 17 | **`USE_TF=0`** injetado nas Etapas 5 e 6 | Bloqueia o carregamento do TensorFlow pelo `transformers`, evitando o crash `runtime_version` no Colab Py3.12 |
| 18 | Reinstalação forçada do `protobuf` (Etapa 7) | Garante que o `onnx2tf` funcione mesmo se o ambiente do Colab corromper a biblioteca |
| 20 | **Auto-Resample (22.05kHz -> 16kHz)** no Piper | Corrige áudios gerados pelo Piper pt_BR que causam o erro `Clip does not have the correct sample rate!` na Etapa 5 |

## Instruções
1. **GPU T4 ativa** (Ambiente de execução → Alterar tipo → T4 GPU)
2. Execute **Etapa 1a** → Clique **Reiniciar sessão** quando solicitado
3. Execute **Etapa 1b em diante** (pode usar "Executar tudo a partir daqui")
4. O modelo `.tflite` será baixado automaticamente no final

> **Tempo total estimado:** ~2-3 horas (inclui download de ~17 GB de features)

---

## Etapa 0: Diagnóstico do Ambiente
Execute primeiro para confirmar a versão do CUDA disponível.

In [ ]:
print('=== DIAGNÓSTICO CUDA ===')
!nvidia-smi | head -10
print()
!nvcc --version 2>/dev/null || echo 'nvcc não encontrado'
print()
print('Bibliotecas CUDA disponíveis:')
!ldconfig -p 2>/dev/null | grep libcudart || find /usr/local/cuda* /usr/lib -name 'libcudart*' 2>/dev/null | head -5
print()
print('=== FIM DO DIAGNÓSTICO ===')

## Etapa 1a: Instalação de Dependências
⚠️ **Reiniciar sessão OBRIGATÓRIO após esta célula!**

In [ ]:
import os, locale, subprocess, sys, re
locale.getpreferredencoding = lambda *a: 'UTF-8'

print('=' * 60)
print('  ETAPA 1a: Instalacao de Dependencias (v20)')
print('=' * 60)

_RUIDO_COLAB = re.compile(
    r'timm|fastai|torchvision|protobuf|numpy|jax|rasterio|shap|'
    r'cupy|opencv|tifffile|grain|tobler|ydf|opentelemetry|grpc|'
    r'google-|xarray|pytensor|transformers|gcsfs|fsspec'
)

def pip_instalar(args, desc='', critico=True):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + \
          (args if isinstance(args, list) else args.split())
    result = subprocess.run(cmd, capture_output=True, text=True)
    saida = result.stdout + result.stderr
    erros, avisos = [], []
    bloco_resolver = False
    for linha in saida.split('\n'):
        l = linha.strip()
        if not l:
            continue
        if "pip's dependency resolver" in l:
            bloco_resolver = True
            continue
        if bloco_resolver:
            if _RUIDO_COLAB.search(l):
                continue
            if 'requires' in l and ('but you have' in l or 'which is not installed' in l):
                avisos.append(l)
            else:
                bloco_resolver = False
        if not bloco_resolver and l.startswith('ERROR:'):
            erros.append(l)
    icone_sym = '❌' if erros else '⚠️' if avisos else '✅'
    if desc:
        print(f'\n{icone_sym} {desc}')
    for a in avisos[:2]:
        print(f'   ⚠️  {a}')
    for e in erros:
        print(f'   ❌ {e}')
    if erros and critico:
        raise RuntimeError(f'Falha critica: {erros[0]}')
    return not erros

def pip_remover(*pkgs):
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'uninstall', '-y'] + list(pkgs),
        capture_output=True, text=True
    )
    removidos = [l.strip() for l in result.stdout.split('\n')
                 if 'Successfully uninstalled' in l]
    for r in removidos:
        print(f'   🗑️  {r}')
    return result.returncode == 0

subprocess.run(['apt-get', 'install', '-qq', 'tree'], capture_output=True)
print('[OK] tree instalado')

for repo, dest in [
    ('https://github.com/dscripka/piper-sample-generator', './piper-sample-generator'),
    ('https://github.com/dscripka/openWakeWord',           './openwakeword'),
]:
    nome = dest.lstrip('./')
    if not os.path.exists(dest):
        subprocess.run(['git', 'clone', '-q', repo, dest], capture_output=True)
        print(f'[OK] {nome} clonado')
    else:
        print(f'[SKIP] {nome}')

print('\n✅ Fase 1: Removendo PyTorch pre-instalado')
pip_remover('torch', 'torchvision', 'torchaudio', 'torchmetrics')

pip_instalar(
    ['torch==2.4.0', 'torchaudio==2.4.0',
     '--index-url', 'https://download.pytorch.org/whl/cu121'],
    desc='Fase 2: PyTorch cu121 (sem torchvision)')

pip_instalar('pathvalidate piper-tts piper-phonemize-cross webrtcvad'.split(),
             desc='Fase 3a: piper/vad')
pip_instalar('mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0'.split(),
             desc='Fase 3b: torchmetrics')
pip_instalar('speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0'.split(),
             desc='Fase 3c: augmentation')
pip_instalar('acoustics==0.2.6 scipy'.split(),
             desc='Fase 3d: acoustics/scipy')
pip_instalar(['datasets>=2.14.0,<3.0.0'],
             desc='Fase 3d2: datasets (pin <3.0 evita transformers 5.x)')
pip_instalar(['transformers>=4.40.0,<4.46.0'],
             desc='Fase 3d3: transformers (pin <4.46 evita flex_attention → beartype crash)')
pip_instalar(['pronouncing'],  desc='Fase 3e: pronouncing')
pip_instalar(['speexdsp-ns'],  desc='Fase 3f: speexdsp-ns', critico=False)
pip_instalar(['-e', './openwakeword', '--no-deps'], desc='Fase 3g: openwakeword --no-deps')

pip_instalar(['onnx>=1.15.0', 'onnxruntime>=1.17.0'],
             desc='Fase 4a: onnx (>=1.15) + onnxruntime (>=1.17)')
pip_instalar(['onnx_graphsurgeon', 'sng4onnx'],
             desc='Fase 4b: onnx utils', critico=False)
pip_instalar(['tensorflow-cpu>=2.16.0', 'keras'],
             desc='Fase 4c: tensorflow-cpu (>=2.16)')
pip_instalar(['onnx2tf'],        desc='Fase 4d: onnx2tf')
pip_instalar(['ai-edge-litert'], desc='Fase 4e: ai-edge-litert', critico=False)

# FIX v19/v20: Desinstalar protobuf antigo e forçar 4.25.5 limpo para TF 2.16+
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'protobuf'], capture_output=True)
pip_instalar(['protobuf==4.25.5', '--no-cache-dir'],
             desc='Fase 4f: protobuf (Fix runtime_version para TF 2.16+)')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r',
     'piper-sample-generator/requirements.txt', '--no-deps'],
    capture_output=True)
print('\n✅ piper-sample-generator deps')

print()
print('=' * 60)
print('  ⚠️  REINICIE A SESSAO AGORA!')
print('  Ambiente de execucao > Reiniciar sessao')
print('  Depois execute a partir da Etapa 1b')
print('=' * 60)

## ⚠️ REINICIAR SESSÃO (OBRIGATÓRIO)

**Ambiente de execução → Reiniciar sessão**

Depois execute a partir da Etapa 1b.

---

## Etapa 1b: Verificação pós-reinício + Downloads

In [ ]:
import os, locale, subprocess, sys, importlib.util
import numpy as np
locale.getpreferredencoding = lambda *a: 'UTF-8'

print('=' * 60)
print('  ETAPA 1b: Verificacao pos-reinicio + Downloads')
print('=' * 60)

_ = np.random.RandomState(42)
print(f'numpy {np.__version__}: OK')

import torch
print(f'PyTorch {torch.__version__}')
if '+cu121' in torch.__version__ or '+cu12' in torch.__version__:
    print('\u2705 CUDA 12.x confirmado')
elif '+cu13' in torch.__version__:
    print('\u274c PyTorch cu13 — execute Etapa 1a novamente!')

try:
    import torchaudio
    print(f'\u2705 torchaudio {torchaudio.__version__}: OK')
except OSError as e:
    print(f'\u274c torchaudio erro: {e}')
    raise

try:
    import torchvision
    tv_ver = torchvision.__version__
    print(f'\u26a0\ufe0f  torchvision {tv_ver} detectado — removendo...')
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'],
                   capture_output=True)
    print('   \u2705 torchvision removido')
except Exception:
    print('\u2705 torchvision nao instalado (correto)')

print('\nVerificando dependencias (sem importar para evitar cadeia onnxscript):')
deps_ok = True
for dep in ['torchmetrics', 'speechbrain', 'audiomentations',
            'torch_audiomentations', 'acoustics', 'pronouncing', 'openwakeword']:
    spec = importlib.util.find_spec(dep)
    if spec is not None:
        print(f'\u2705 {dep}: instalado')
    else:
        print(f'\u274c {dep}: FALTANDO — execute Etapa 1a novamente')
        deps_ok = False

try:
    import importlib.metadata
    tv = importlib.metadata.version('transformers')
    major = int(tv.split('.')[0])
    if major >= 5:
        print(f'\u26a0\ufe0f  transformers {tv} >= 5.0 — pode causar conflito!')
        print('   Execute: pip install transformers>=4.40.0,<5.0.0 e reinicie')
        deps_ok = False
    else:
        print(f'\u2705 transformers {tv} < 5.0: OK')
except Exception:
    pass

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'\u2705 GPU: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('\u26a0\ufe0f  GPU nao disponivel')

if not deps_ok:
    raise RuntimeError('Dependencias faltando ou incompativeis! Execute Etapa 1a e reinicie.')

models_dir = 'openwakeword/openwakeword/resources/models'
os.makedirs(models_dir, exist_ok=True)
base_url = 'https://github.com/dscripka/openWakeWord/releases/download/v0.5.1'
for fname in ['embedding_model.onnx', 'embedding_model.tflite',
              'melspectrogram.onnx', 'melspectrogram.tflite']:
    fpath = os.path.join(models_dir, fname)
    if not os.path.exists(fpath):
        subprocess.run(['wget', '-q', f'{base_url}/{fname}', '-O', fpath], capture_output=True)
        print(f'[OK] {fname}')
    else:
        print(f'[SKIP] {fname}')

libritts = 'piper-sample-generator/models/en_US-libritts_r-medium.pt'
if not os.path.exists(libritts):
    os.makedirs('piper-sample-generator/models', exist_ok=True)
    url = 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
    print('Baixando LibriTTS v2...')
    subprocess.run(['wget', '-q', '-O', libritts, url], capture_output=True)
    print('[OK] LibriTTS v2')
else:
    print('[SKIP] LibriTTS')

os.makedirs('piper_voices_ptbr', exist_ok=True)
hf_base = 'https://huggingface.co/rhasspy/piper-voices/resolve/main'
for name, path in [('pt_BR-faber-medium', 'pt/pt_BR/faber/medium'),
                   ('pt_BR-edresson-low',  'pt/pt_BR/edresson/low')]:
    onnx_path = f'piper_voices_ptbr/{name}.onnx'
    if not os.path.exists(onnx_path):
        subprocess.run(['wget', '-q', '-O', onnx_path,
                        f'{hf_base}/{path}/{name}.onnx'], capture_output=True)
        subprocess.run(['wget', '-q', '-O', f'{onnx_path}.json',
                        f'{hf_base}/{path}/{name}.onnx.json'], capture_output=True)
        print(f'[OK] {name}')
    else:
        print(f'[SKIP] {name}')

os.makedirs('output/etapa-01', exist_ok=True)
r = subprocess.run(['tree', '-L', '3', '--noreport', '/content'], capture_output=True, text=True)
if r.returncode != 0:
    r = subprocess.run(['find', '/content', '-maxdepth', '3', '-not', '-path', '*/.*'],
                       capture_output=True, text=True)
with open('output/etapa-01/tree.txt', 'w') as f:
    f.write(r.stdout)
print(f'\n\U0001f4c1 Arvore salva: output/etapa-01/tree.txt')
print('[OK] ETAPA 1b CONCLUIDA!')

## Etapa 2: Testar Pronúncia
Ouça os áudios. Confirme que "Letícia" soa correto em pt_BR.

In [ ]:
import subprocess, os
from IPython.display import Audio, display

print('=' * 60)
print('  ETAPA 2: Testando pronúncia pt_BR')
print('=' * 60)

target_word = 'letícia'
os.makedirs('test_audio', exist_ok=True)

for name, label in [('pt_BR-faber-medium', 'Faber'), ('pt_BR-edresson-low', 'Edresson')]:
    out = f'test_audio/test_{label.lower()}.wav'
    r = subprocess.run(
        ['piper', '--model', f'piper_voices_ptbr/{name}.onnx', '--output_file', out],
        input=target_word, capture_output=True, text=True
    )
    if os.path.exists(out):
        print(f'Voz {label} (pt_BR):')
        display(Audio(out, autoplay=False))
    else:
        print(f'[ERRO] {label}: {r.stderr[:200]}')

import subprocess
os.makedirs('output/etapa-02', exist_ok=True)
r = subprocess.run(['tree', '-L', '3', '--noreport', '/content'],
                   capture_output=True, text=True)
if r.returncode != 0:
    r = subprocess.run(['find', '/content', '-maxdepth', '3', '-not', '-path', '*/.*'],
                       capture_output=True, text=True)
with open('output/etapa-02/tree.txt', 'w') as f:
    f.write(r.stdout)
print(f'\n📁 Árvore salva: output/etapa-02/tree.txt')
print('[OK] ETAPA 2 CONCLUÍDA!')

## Etapa 3: Download de Dados Auxiliares

| Dado | Tamanho | Tempo estimado |
|------|---------|----------------|
| MIT RIRs | ~30 MB | ~2 min |
| AudioSet | ~1 GB | ~5 min |
| FMA | ~300 MB | ~2 min |
| ACAV100M features | **17.3 GB** | ~10-20 min |
| Validation features | 185 MB | ~1 min |

In [ ]:
import os
import numpy as np
import scipy.io.wavfile as wav
import datasets
import torchaudio
from tqdm.auto import tqdm
from pathlib import Path

print('=' * 60)
print('  ETAPA 3: Baixando dados auxiliares')
print('=' * 60)

rir_dir = 'mit_rirs'
if not os.path.exists(rir_dir) or len([f for f in os.listdir(rir_dir) if f.endswith('.wav')]) == 0:
    print('\n[3a] MIT Room Impulse Responses...')
    os.makedirs(rir_dir, exist_ok=True)
    rir_dataset = datasets.load_dataset(
        'davidscripka/MIT_environmental_impulse_responses',
        split='train', streaming=True
    )
    count = 0
    for row in tqdm(rir_dataset, desc='MIT RIRs'):
        name = row['audio']['path'].split('/')[-1]
        audio_array = np.array(row['audio']['array'])
        wav.write(os.path.join(rir_dir, name), 16000,
                  (audio_array * 32767).astype(np.int16))
        count += 1
    print(f'[OK] MIT RIRs: {count} arquivos')
else:
    n = len([f for f in os.listdir(rir_dir) if f.endswith('.wav')])
    print(f'[SKIP] MIT RIRs ({n} arquivos)')

as_dir = 'audioset_16k'
if not os.path.exists(as_dir) or len(os.listdir(as_dir)) == 0:
    print('\n[3b] AudioSet background noise...')
    os.makedirs('audioset', exist_ok=True)
    os.makedirs(as_dir, exist_ok=True)
    fname = 'bal_train09.tar'
    link = f'https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/{fname}'
    if not os.path.exists(f'audioset/{fname}'):
        !wget -q --show-progress -O audioset/{fname} '{link}'
    !cd audioset && tar -xf {fname} 2>/dev/null || true
    flac_files = list(Path('audioset/audio').glob('**/*.flac')) if os.path.exists('audioset/audio') else []
    if flac_files:
        print(f'  Convertendo {len(flac_files)} FLAC → WAV 16kHz...')
        audioset_ds = datasets.Dataset.from_dict({'audio': [str(i) for i in flac_files]})
        audioset_ds = audioset_ds.cast_column('audio', datasets.Audio(sampling_rate=16000))
        for row in tqdm(audioset_ds, desc='AudioSet'):
            name = row['audio']['path'].split('/')[-1].replace('.flac', '.wav')
            audio_array = np.array(row['audio']['array'])
            wav.write(os.path.join(as_dir, name), 16000,
                      (audio_array * 32767).astype(np.int16))
    n = len(os.listdir(as_dir))
    print(f'[OK] AudioSet: {n} clips')
else:
    print(f'[SKIP] AudioSet ({len(os.listdir(as_dir))} clips)')

fma_dir = 'fma'
if not os.path.exists(fma_dir) or len(os.listdir(fma_dir)) == 0:
    print('\n[3c] FMA música (1 hora)...')
    os.makedirs(fma_dir, exist_ok=True)
    try:
        fma_dataset = datasets.load_dataset(
            'rudraml/fma', name='small', split='train',
            streaming=True, trust_remote_code=True
        )
        fma_iter = iter(fma_dataset.cast_column('audio', datasets.Audio(sampling_rate=16000)))
        n_clips = 120
        count = 0
        failures = 0
        for i in tqdm(range(n_clips), desc='FMA'):
            try:
                row = next(fma_iter)
                name = row['audio']['path'].split('/')[-1].replace('.mp3', '.wav')
                audio_array = np.array(row['audio']['array'])
                wav.write(os.path.join(fma_dir, name), 16000,
                          (audio_array * 32767).astype(np.int16))
                count += 1
                failures = 0
            except StopIteration:
                break
            except (ValueError, Exception) as e:
                failures += 1
                if failures >= 5:
                    print(f'\n  ⚠️  FMA: {failures} falhas consecutivas ({type(e).__name__}). '
                          'Abortando — AudioSet cobre o background.')
                    break
                continue
        if count > 0:
            print(f'[OK] FMA: {count} clips')
        else:
            print('[AVISO] FMA: 0 clips obtidos — continuando sem FMA (AudioSet cobre o background)')
    except Exception as e:
        print(f'[AVISO] FMA falhou ao carregar ({type(e).__name__}: {e})')
        print('  Continuando sem FMA — AudioSet será usado como background')
else:
    print(f'[SKIP] FMA ({len(os.listdir(fma_dir))} clips)')

print('\n[FIX 4] Verificando sample rate dos clips FMA...')
fma_wav = [f for f in os.listdir(fma_dir) if f.endswith('.wav')] if os.path.exists(fma_dir) else []
bad_sr = []
for fname in fma_wav:
    fpath = os.path.join(fma_dir, fname)
    try:
        info = torchaudio.info(fpath)
        if info.sample_rate != 16000:
            bad_sr.append((fname, info.sample_rate))
    except Exception:
        pass
if bad_sr:
    print(f'  ⚠️  {len(bad_sr)} clips com sample rate incorreto — corrigindo para 16kHz...')
    for fname, sr in tqdm(bad_sr, desc='Resample FMA'):
        fpath = os.path.join(fma_dir, fname)
        waveform, _ = torchaudio.load(fpath)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
        torchaudio.save(fpath, waveform, 16000)
    print(f'  ✅ {len(bad_sr)} clips corrigidos para 16kHz')
elif fma_wav:
    print(f'  ✅ Todos os {len(fma_wav)} clips FMA já estão em 16kHz')
else:
    print('  ℹ️  Nenhum clip FMA — etapa pulada')

acav_file = 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
if not os.path.exists(acav_file):
    print('\n[3d] ACAV100M features (17.3 GB — ~10-20 min)...')
    url = 'https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
    !wget -q --show-progress -c '{url}'
    if os.path.exists(acav_file):
        size_gb = os.path.getsize(acav_file) / (1024**3)
        print(f'[OK] ACAV100M: {size_gb:.1f} GB')
    else:
        print('[ERRO] ACAV100M download falhou!')
else:
    size_gb = os.path.getsize(acav_file) / (1024**3)
    print(f'[SKIP] ACAV100M ({size_gb:.1f} GB)')

val_file = 'validation_set_features.npy'
if not os.path.exists(val_file):
    print('\n[3e] Validation features (185 MB)...')
    url = 'https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy'
    !wget -q --show-progress '{url}'
    print('[OK] Validation features')
else:
    size_mb = os.path.getsize(val_file) / (1024**2)
    print(f'[SKIP] Validation features ({size_mb:.0f} MB)')

print()
print('Resumo dos dados:')
for d, label in [('mit_rirs', 'RIRs'), ('audioset_16k', 'AudioSet'), ('fma', 'FMA')]:
    if os.path.exists(d):
        n = len([f for f in os.listdir(d) if f.endswith('.wav')])
        print(f'  {label}: {n} arquivos WAV')
    else:
        print(f'  ❌ {label}: FALTANDO')
for f, label in [(acav_file, 'ACAV100M'), (val_file, 'Validation')]:
    if os.path.exists(f):
        s = os.path.getsize(f) / (1024**2)
        unit = 'GB' if s > 1024 else 'MB'
        val = s/1024 if s > 1024 else s
        print(f'  {label}: {val:.1f} {unit}')
    else:
        print(f'  ❌ {label}: FALTANDO')

import subprocess
os.makedirs('output/etapa-03', exist_ok=True)
r = subprocess.run(['tree', '-L', '3', '--noreport', '/content'],
                   capture_output=True, text=True)
if r.returncode != 0:
    r = subprocess.run(['find', '/content', '-maxdepth', '3', '-not', '-path', '*/.*'],
                       capture_output=True, text=True)
with open('output/etapa-03/tree.txt', 'w') as f:
    f.write(r.stdout)
print(f'\n📁 Árvore salva: output/etapa-03/tree.txt')
print('[OK] ETAPA 3 CONCLUÍDA!')

## Etapa 4: Gerar Clips com Piper pt_BR

In [ ]:
import os, subprocess, uuid, random, time
from concurrent.futures import ThreadPoolExecutor, as_completed

print('=' * 60)
print('  ETAPA 4: Gerando clips com Piper pt_BR')
print('=' * 60)

target_word = 'letícia'
model_name  = 'leticia'
n_positive  = 1500
n_val       = 500

ptbr_voices = [
    'piper_voices_ptbr/pt_BR-faber-medium.onnx',
    'piper_voices_ptbr/pt_BR-edresson-low.onnx',
]
length_scales = [0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2, 1.25]
noise_scales  = [0.5, 0.6, 0.667, 0.7, 0.8, 0.9, 0.98]
noise_ws      = [0.5, 0.6, 0.7, 0.8, 0.9, 0.98]

negative_words = [
    'patrícia', 'notícia', 'delícia', 'justiça', 'preguiça',
    'milícia', 'malícia', 'polícia', 'carência', 'urgência',
    'letivo', 'letrada', 'legítima', 'legião', 'elétrica',
    'lícia', 'alícia', 'felícia', 'luciana', 'larissa',
    'olá', 'bom dia', 'boa noite', 'obrigado', 'por favor',
    'ligar', 'desligar', 'acender', 'apagar', 'aumentar',
    'diminuir', 'temperatura', 'música', 'que horas são',
    'televisão', 'computador', 'celular', 'internet', 'cozinha',
]

base = f'./my_custom_model/{model_name}'
dirs = {
    'positive_train': f'{base}/positive_train',
    'positive_test':  f'{base}/positive_test',
    'negative_train': f'{base}/negative_train',
    'negative_test':  f'{base}/negative_test',
}
for d in dirs.values():
    os.makedirs(d, exist_ok=True)

def gen_one_clip(args):
    word, voice, out_path = args
    try:
        subprocess.run(
            ['piper', '--model', voice, '--output_file', out_path,
             '--length-scale', str(random.choice(length_scales)),
             '--noise-scale',  str(random.choice(noise_scales)),
             '--noise-w',      str(random.choice(noise_ws))],
            input=word, capture_output=True, text=True, timeout=30
        )
        return os.path.exists(out_path)
    except Exception:
        return False

def gen_clips_parallel(text, out_dir, n, label, workers=4):
    existing = len([f for f in os.listdir(out_dir) if f.endswith('.wav')])
    if existing >= int(n * 0.95):
        print(f'  [SKIP] {label}: {existing} clips já existem')
        return
    needed = n - existing
    texts = [text] if isinstance(text, str) else text
    tasks = []
    for i in range(needed):
        word  = random.choice(texts) if isinstance(texts, list) else texts
        voice = random.choice(ptbr_voices)
        out   = os.path.join(out_dir, f'{uuid.uuid4().hex}.wav')
        tasks.append((word, voice, out))
    count = 0
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(gen_one_clip, t): t for t in tasks}
        for future in as_completed(futures):
            if future.result():
                count += 1
            if count % 200 == 0 and count > 0:
                elapsed = time.time() - t0
                rate = count / elapsed
                remaining = (needed - count) / rate if rate > 0 else 0
                print(f'  {label}: {count}/{needed} ({rate:.1f} clips/s, ~{remaining/60:.0f} min)')
    total = len([f for f in os.listdir(out_dir) if f.endswith('.wav')])
    print(f'  [OK] {label}: {total} clips ({(time.time()-t0)/60:.1f} min)')

N_WORKERS = 4
print(f'Usando {N_WORKERS} workers paralelos\n')
gen_clips_parallel(target_word,    dirs['positive_train'], n_positive, 'Positivos treino', N_WORKERS)
gen_clips_parallel(target_word,    dirs['positive_test'],  n_val,      'Positivos val',    N_WORKERS)
gen_clips_parallel(negative_words, dirs['negative_train'], n_positive, 'Negativos treino', N_WORKERS)
gen_clips_parallel(negative_words, dirs['negative_test'],  n_val,      'Negativos val',    N_WORKERS)

print('\nResumo:')
for label, d in dirs.items():
    n = len([f for f in os.listdir(d) if f.endswith('.wav')])
    status = '✅' if n >= int({'positive_train': n_positive, 'positive_test': n_val,
                               'negative_train': n_positive, 'negative_test': n_val}[label] * 0.95) else '⚠️'
    print(f'  {status} {label}: {n} clips')

import subprocess
os.makedirs('output/etapa-04', exist_ok=True)
r = subprocess.run(['tree', '-L', '4', '--noreport', '/content/my_custom_model'],
                   capture_output=True, text=True)
if r.returncode != 0:
    r = subprocess.run(['find', '/content/my_custom_model', '-not', '-path', '*/.*'],
                       capture_output=True, text=True)
with open('output/etapa-04/tree.txt', 'w') as f:
    f.write(r.stdout)
print(f'\n📁 Árvore salva: output/etapa-04/tree.txt')
print('[OK] ETAPA 4 CONCLUÍDA!')

## Etapa 5: Augmentação + Extração de Features

**Fixes aplicados nesta etapa:**
- **Fix 5**: Smart cleanup — detecta features incompletos e remove apenas `.npy` (preserva clips de treino)
- **Fix 17**: `USE_TF=0` desativa o carregamento acidental do TensorFlow e evita crash no Colab.
- **Fix 20**: O Piper usa 22050Hz nas vozes Medium. A ferramenta fará Auto-Resample para 16000Hz nos áudios sintéticos.

In [ ]:
import os, sys, yaml, glob, importlib.util
import numpy as np
import torchaudio
from tqdm.auto import tqdm

# ── FIX 17: Evitar importação acidental do TensorFlow via transformers ──
os.environ['USE_TF'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

print('=' * 60)
print('  ETAPA 5: Augmentação e extração de features')
print('=' * 60)

# ── FIX 13: Pre-flight usa find_spec em vez de __import__ ─────
print('\n[Pre-flight] Verificando dependências (sem importar para evitar cadeia onnxscript)...')
preflight_ok = True
for dep in ['torchmetrics', 'speechbrain', 'audiomentations',
            'torch_audiomentations', 'acoustics', 'pronouncing']:
    spec = importlib.util.find_spec(dep)
    if spec is not None:
        print(f'  ✅ {dep}: instalado')
    else:
        print(f'  ❌ {dep} — execute: pip install {dep}')
        preflight_ok = False

if not preflight_ok:
    raise ImportError('Dependências faltando! Execute Etapa 1a, reinicie e tente novamente.')

model_name = 'leticia'

# ── FIX 20: Corrigir Sample Rate dos clips do Piper (22050Hz -> 16000Hz) ──
print('\n[FIX 20] Verificando Sample Rate dos áudios gerados pelo Piper...')
clip_dirs = [
    f'my_custom_model/{model_name}/positive_train',
    f'my_custom_model/{model_name}/positive_test',
    f'my_custom_model/{model_name}/negative_train',
    f'my_custom_model/{model_name}/negative_test'
]

for d in clip_dirs:
    if not os.path.exists(d): continue
    wavs = [f for f in os.listdir(d) if f.endswith('.wav')]
    bad_clips = []
    for w in wavs:
        p = os.path.join(d, w)
        try:
            info = torchaudio.info(p)
            if info.sample_rate != 16000:
                bad_clips.append((p, info.sample_rate))
        except Exception: 
            pass
            
    if bad_clips:
        print(f'  ⚠️ {len(bad_clips)} clips em {os.path.basename(d)} não estão em 16kHz. Convertendo...')
        for p, sr in tqdm(bad_clips, desc=f'Resample {os.path.basename(d)}'):
            wf, _ = torchaudio.load(p)
            if wf.shape[0] > 1: wf = wf.mean(dim=0, keepdim=True)
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
            wf_16k = resampler(wf)
            torchaudio.save(p, wf_16k, 16000)
        print(f'  ✅ {os.path.basename(d)} convertido para 16kHz.')
    else:
        print(f'  ✅ {os.path.basename(d)} já está em 16kHz.')

# ── FIX 5: Smart cleanup de features ─────────────────────────
feature_dir = f'my_custom_model/{model_name}'
os.makedirs(feature_dir, exist_ok=True)
expected_features = [
    'positive_features_train.npy', 'negative_features_train.npy',
    'positive_features_test.npy',  'negative_features_test.npy',
]
existing = [f for f in expected_features
            if os.path.exists(os.path.join(feature_dir, f))]

print(f'\n[Smart cleanup] Features existentes: {len(existing)}/{len(expected_features)}')
if 0 < len(existing) < len(expected_features):
    print(f'  ⚠️  Features incompletos — removendo parciais (clips preservados)...')
    for f in existing:
        fp = os.path.join(feature_dir, f)
        os.remove(fp)
        print(f'  🗑️  Removido: {f}')
    print(f'  ✅ {len(existing)} features parciais removidos. Clips preservados.')
elif len(existing) == len(expected_features):
    print(f'  ✅ Features completos — train.py usará cache existente')
else:
    print(f'  ℹ️  Nenhum feature existente — gerando do zero')

# ── Configuração ───────────────────────────────────────────────
if os.path.exists('mit_rirs') and len([f for f in os.listdir('mit_rirs') if f.endswith('.wav')]) > 0:
    rir_path = os.path.abspath('mit_rirs')
else:
    rir_path = os.path.abspath('piper-sample-generator/impulses')
n_rir = len(os.listdir(rir_path))
print(f'\nRIR: {rir_path} ({n_rir} arquivos)')

audioset_path = os.path.abspath('./audioset_16k')
fma_path      = os.path.abspath('./fma')
n_audioset = len([f for f in os.listdir(audioset_path) if f.endswith('.wav')]) if os.path.exists(audioset_path) else 0
n_fma      = len([f for f in os.listdir(fma_path)      if f.endswith('.wav')]) if os.path.exists(fma_path)      else 0
print(f'Background: AudioSet={n_audioset} clips | FMA={n_fma} clips')

background_paths = []
if n_audioset > 0:
    background_paths.append(audioset_path)
if n_fma > 0:
    background_paths.append(fma_path)
if not background_paths:
    print('⚠️  Sem dados de background! Usando RIRs como fallback')
    background_paths = [rir_path]

config = {
    'model_name': model_name,
    'target_phrase': ['letícia'],
    'custom_negative_phrases': [
        'patrícia', 'notícia', 'delícia', 'justiça',
        'milícia', 'malícia', 'polícia', 'alícia', 'felícia'
    ],
    'n_samples': 1500,
    'n_samples_val': 500,
    'tts_batch_size': 50,
    'augmentation_batch_size': 16,
    'piper_sample_generator_path': os.path.abspath('./piper-sample-generator'),
    'output_dir': os.path.abspath('./my_custom_model'),
    'rir_paths': [rir_path],
    'background_paths': background_paths,
    'background_paths_duplication_rate': [1] * len(background_paths),
    'augmentation_rounds': 1,
    'false_positive_validation_data_path': os.path.abspath('./validation_set_features.npy'),
    'feature_data_files': {
        'ACAV100M_sample': os.path.abspath('./openwakeword_features_ACAV100M_2000_hrs_16bit.npy')
    },
    'batch_n_per_class': {
        'ACAV100M_sample': 1024,
        'adversarial_negative': 50,
        'positive': 50
    },
    'model_type': 'dnn',
    'layer_size': 32,
    'steps': 50000,
    'max_negative_weight': 1500,
    'target_false_positives_per_hour': 0.2,
}

with open('my_model.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)
print('Config salva: my_model.yaml')

# ── FIX 15: Patch onnxscript/values.py em disco ───────────────
import inspect as _inspect, glob as _glob

print('\n[FIX 15] Aplicando patch onnxscript.values.ParamSchema em disco...')
try:
    import onnxscript.values as _ov
    _ov_path = _inspect.getfile(_ov)
    if _ov_path.endswith('.pyc'):
        _ov_path = _ov_path.replace('/__pycache__/', '/').rsplit('.', 2)[0] + '.py'
    print(f'  onnxscript/values.py: {_ov_path}')
    if not hasattr(_ov, 'ParamSchema'):
        with open(_ov_path, 'a') as _f:
            _f.write(
                '\n# compat-patch: restore ParamSchema for torch 2.4.0\n'
                '# (removida em onnxscript recente; beartype em op_validation.py ainda referencia)\n'
                'class ParamSchema:\n'
                '    pass\n'
            )
        _pyc_dir = os.path.join(os.path.dirname(_ov_path), '__pycache__')
        for _pyc in _glob.glob(os.path.join(_pyc_dir, 'values*.pyc')):
            os.remove(_pyc)
            print(f'  🗑️  .pyc removido: {os.path.basename(_pyc)}')
        print(f'  ✅ Patch aplicado: class ParamSchema injetada em {os.path.basename(_ov_path)}')
    else:
        print('  ✅ ParamSchema já existe — patch não necessário')
except Exception as _e:
    print(f'  ⚠️  Patch falhou ({type(_e).__name__}: {_e})')

# ── Augmentação ────────────────────────────────────────────────
!USE_TF=0 {sys.executable} openwakeword/openwakeword/train.py \
    --training_config my_model.yaml \
    --augment_clips

# ── Verificação dos features gerados ──────────────────────────
print()
print('Features gerados:')
all_ok = True
for f in expected_features:
    fp = os.path.join(feature_dir, f)
    if os.path.exists(fp):
        shape = np.load(fp, mmap_mode='r').shape
        print(f'  ✅ {f}: {shape}')
    else:
        print(f'  ❌ {f}: NÃO ENCONTRADO')
        all_ok = False

import subprocess
os.makedirs('output/etapa-05', exist_ok=True)
r = subprocess.run(['tree', '-L', '3', '--noreport', '/content/my_custom_model'],
                   capture_output=True, text=True)
if r.returncode != 0:
    r = subprocess.run(['find', '/content/my_custom_model', '-not', '-path', '*/.*'],
                       capture_output=True, text=True)
with open('output/etapa-05/tree.txt', 'w') as f:
    f.write(r.stdout)
print(f'\n📁 Árvore salva: output/etapa-05/tree.txt')

if all_ok:
    print('\n[OK] ETAPA 5 CONCLUÍDA!')
else:
    print('\n[ERRO] Alguns features não foram gerados. Verifique os logs acima.')

## Etapa 6: Treinar o Modelo
**Tempo estimado: ~30-60 min com GPU T4**

In [ ]:
import sys, os

print('=' * 60)
print('  ETAPA 6: Treinando modelo')
print('=' * 60)

os.environ['USE_TF'] = '0'

!USE_TF=0 {sys.executable} openwakeword/openwakeword/train.py \
    --training_config my_model.yaml \
    --train_model

import subprocess
os.makedirs('output/etapa-06', exist_ok=True)
r = subprocess.run(['tree', '-L', '3', '--noreport', '/content/my_custom_model'],
                   capture_output=True, text=True)
if r.returncode != 0:
    r = subprocess.run(['find', '/content/my_custom_model', '-not', '-path', '*/.*'],
                       capture_output=True, text=True)
with open('output/etapa-06/tree.txt', 'w') as f:
    f.write(r.stdout)
print(f'\n📁 Árvore salva: output/etapa-06/tree.txt')
print('[OK] ETAPA 6 CONCLUÍDA!')

## Etapa 7: Converter para TFLite e Baixar
**Usa `onnx2tf` + `tensorflow-cpu==2.16+` — estável no Colab.**

In [ ]:
import os, glob, shutil, subprocess, sys
from google.colab import files

print('=' * 60)
print('  ETAPA 7: Modelo final')
print('=' * 60)

# Remove a restrição do TensorFlow que usamos nas etapas 5 e 6
if 'USE_TF' in os.environ:
    del os.environ['USE_TF']

# FIX 18: Workaround para erro de protobuf no onnx2tf e Colab Py3.12
try:
    import tensorflow as tf
    print('✅ TensorFlow validado para a conversão onnx2tf.')
except ImportError as e:
    print(f'⚠️ Falha na importação do TensorFlow: {e}')
    print('   Limpando e restaurando protobuf antes de continuar...')
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'protobuf'], capture_output=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'protobuf==4.25.5', '--no-cache-dir'], capture_output=True)
    print('   ✅ Protobuf restaurado.')
    import tensorflow as tf

model_name  = 'leticia'
onnx_path   = f'my_custom_model/{model_name}.onnx'
tflite_path = f'my_custom_model/{model_name}.tflite'

if os.path.exists(onnx_path) and not os.path.exists(tflite_path):
    print('TFLite não gerado. Convertendo via onnx2tf...')
    tf_out = f'my_custom_model/tf_model_{model_name}'
    !onnx2tf -i {onnx_path} -o {tf_out} -oiqt 2>&1 | tail -20
    tflite_files = glob.glob(f'{tf_out}/**/*.tflite', recursive=True)
    if tflite_files:
        float_models = [f for f in tflite_files if 'float32' in f or 'float16' in f]
        chosen = float_models[0] if float_models else tflite_files[0]
        shutil.copy2(chosen, tflite_path)
        print(f'[OK] TFLite gerado: {os.path.basename(chosen)}')
    else:
        print('[ERRO] Conversão TFLite falhou!')

print()
for path, label in [(onnx_path, 'ONNX'), (tflite_path, 'TFLite')]:
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        print(f'  ✅ {label}: {path} ({size:.1f} KB)')
    else:
        print(f'  ❌ {label}: não encontrado')

print()
for path in [tflite_path, onnx_path]:
    if os.path.exists(path):
        print(f'Baixando {os.path.basename(path)}...')
        files.download(path)

import subprocess
os.makedirs('output/etapa-07', exist_ok=True)
r = subprocess.run(['tree', '-L', '3', '--noreport', '/content'],
                   capture_output=True, text=True)
if r.returncode != 0:
    r = subprocess.run(['find', '/content', '-maxdepth', '3', '-not', '-path', '*/.*'],
                       capture_output=True, text=True)
with open('output/etapa-07/tree.txt', 'w') as f:
    f.write(r.stdout)
print(f'\n📁 Árvore salva: output/etapa-07/tree.txt')

print()
print('=' * 60)
print('  🎉 CONCLUÍDO!')
print()
print('  Deploy no Home Assistant Yellow:')
print('  1. Copie leticia.tflite → /share/openwakeword/')
print('  2. Reinicie o add-on openWakeWord')
print('  3. Configure: Assistants > Wake Word > leticia')
print('  4. Teste: "Letícia, que horas são?"')
print('=' * 60)